# Costruiamo una pipeline ETL: da CSV a database SQLite

1- Importiamo tre serie storiche di mercato (indice azionario mondiale, indice azionario
mondiale small cap, oro) da file CSV a un database SQLite, con una pipeline in tre
fasi: **Extract → Transform → Load**.

2- Interroghiamo il database con SQL\Python.

**Obiettivo**: ripassare pandas (lettura CSV, allineamento di serie temporali,
gestione dei dati mancanti) e imparare i concetti base di una pipeline ETL, incluso
come evitare duplicati quando la si rilancia più volte.


## Setup

Importiamo le librerie che useremo.

In [ ]:
import pandas as pd
import sqlite3


## 1) Extract

Ogni file CSV ha due colonne: la prima è la data, la seconda il valore. L'obiettivo è unire tutte le fonti in un unico DataFrame in formato "wide" (una riga per data, una colonna per fonte), allineate cronologicamente.

In [ ]:
url = "https://raw.githubusercontent.com/GabrieleL98/ADABI_Python_LAB/main/"

elenco = {
    "WORLD": "world.csv",
    "WORLD_SMALL_CAP": "world_small_cap.csv",
    "GOLD": "gold.csv"
    }

#il nome del nostro futuro db
db_path = "mercati.db"

### GUIDA e SUGGERIMENTI

La soluzione intuitiva per chi ha familiarità con i database relazionali si basa su una serie di `join` (o `merge`) consecutivi.

Pandas, tuttavia, offre una scorciatoia basata sull'allineamento automatico degli indici: passando a `pd.DataFrame()` un dizionario di Series aventi lo stesso indice (*hint: `set_index`*), la tabella verrà costruita e allineata automaticamente lungo le date.

Scegliete l'opzione che preferite, o provatene di nuove.

### Indicazioni pratiche:

* **`pd.read_csv`** accetta come argomento il percorso al file (es. `"cartella/file.csv"` o un URL web). L'url alla repository in cui si trovano i file ve l'ho lasciato nella cella qua sopra, così come l'elenco con i nomi dei csv da importare presenti in quella repository
* Parametri utili da impostare in `read_csv`:
  1. **`usecols=[0,1]**: indica a Pandas di caricare solo le prime due colonne, ignorando eventuali dati superflui.
  2. **`header=0`**: indica che la riga 0 del CSV contiene l'intestazione originale che vogliamo rimpiazzare (è il nostro caso).
  3. **`names=["X", "y", "Z" ...]`**: lista con i nuovi nomi da assegnare alle colonne caricate.
  4. **`parse_dates=[]`**: forza la conversione automatica della colonna specificata nel tipo `datetime64`.

**SCRIVI UNA FUNZIONE DI EXTRACT**

In [ ]:
def extract(elenco, url):
    """
    Legge ogni file CSV elencato in `elenco` (nome_colonna -> percorso_file) e le
    unisce in un solo DataFrame "wide": una riga per data, una colonna per fonte.
    """

    return df_raw



## 2) Transform

Le fonti non hanno tutte le stesse date (festivi diversi, giorni mancanti).
Costruiamo un calendario giornaliero continuo (tutti i giorni, weekend compresi)
e riportiamo avanti (*forward-fill*) l'ultimo valore conosciuto per riempire i
buchi — es. un sabato prende il valore del venerdì precedente.

**Attenzione ai NaN iniziali**: GOLD ha dati dal 1998, ma WORLD e WORLD_SMALL_CAP
solo dal 2000. Per quelle prime righe, WORLD e WORLD_SMALL_CAP resteranno `NaN`
anche dopo il forward-fill (non c'è "un valore precedente" da riportare avanti,
perché quell'indice semplicemente non esisteva ancora).

Indicazioni pratiche:

*   pandas ha una funzione chamata **`data_range()`** che prende come argomenti una data di inizio e una di fine, e la frequenza ('D') e crea una calendario continuo tra quelle due date
*   **`reindex()`** forza una serie (che prende come argomento) come nuovo indice di un DF, mettendo a le nuove righe aggiunte.
*   **`ffill()`** fa esattamente ciò che cerchiamo, ovvero propagare in avanti l'ultimo valore trovato.

N.B. `ffill()` è un metodo, non una funzione perciò la sua applicazione è `x.ffill()` e non `ffill(x)`

**SCRIVI UNA FUNZIONE DI TRASFORM**

In [ ]:
def transform(df_combined):
    """
    Crea un calendario giornaliero continuo tra la prima e l'ultima data
    disponibile, e riporta avanti (ffill) l'ultimo valore noto per riempire i
    giorni mancanti.
    """

    return df_clean


## 3) Load

Scriviamo `df_clean` in una tabella SQLite chiamata `prices`. Punto chiave: se
rilanciamo lo script più volte NON vogliamo che le righe si accumulino
all'infinito. Per questo:

- la colonna `Date` è **PRIMARY KEY**: SQLite non può avere due righe con la
  stessa data;
- scriviamo con **`INSERT OR REPLACE`**: se la data esiste già, la riga viene
  sovrascritta con i nuovi valori; altrimenti viene creata.

Risultato atteso: rilanciare lo script quante volte vogliamo produce sempre lo stesso
identico contenuto nel database. Nessun duplicato o crescita infinita.

Per semplicità, dato che SQLite non è l'obiettivo di questo laboratorio, il codice è già scritto. Vale comunque la pena spenderci del tempo per capirne il funzionamento.


In [ ]:
df_clean = transform(extract(url, elenco))

df_clean.index.name = 'Date'

#il nome del nostro db
db_path = "mercati.db"

def load(df_final, db_path):
    conn = sqlite3.connect(db_path)

    conn.execute("""
        CREATE TABLE IF NOT EXISTS prices (
            Date TEXT PRIMARY KEY,
            WORLD REAL,
            WORLD_SMALL_CAP REAL,
            GOLD REAL
        )
    """)

    # rimette il classico indice 0, 1, 2... trasformando l'indice "data" a colonna
    df_da_scrivere = df_clean.reset_index()

    # trasforma il formato data in formato standard ISO SQLite AAAA-MM-DD
    df_da_scrivere['Date'] = df_da_scrivere['Date'].dt.strftime('%Y-%m-%d')

    # operazione di insert righe
    conn.executemany(
        """
        INSERT OR REPLACE INTO prices (Date, WORLD, WORLD_SMALL_CAP, GOLD)
        VALUES (?, ?, ?, ?)
        """,
        # (?, ? , ?, ?) è un placeholder necessario per il funzionamento della query
        # trasforma le colonne in una matrice numpy con "[[]]"" e poi in liste, step necessario per passarle come VALUES a SQL
        df_da_scrivere[['Date', 'WORLD', 'WORLD_SMALL_CAP', 'GOLD']].values.tolist(),
    )

    conn.commit()
    conn.close()

load(df_clean, db_path)
print(f"Primo funzione: Caricate {len(df_clean)} righe nel database {db_path}.")


NameError: name 'transform' is not defined

In [ ]:
# Oppure, più breve ma meno "parlante":
def load(df_final, db_path):
    conn = sqlite3.connect(db_path)
    df_final.to_sql("prices", conn, if_exists="replace", index=True, index_label="Date")
    conn.close()

load(df_clean, db_path)
print(f"Seconda funzione: Caricate {len(df_clean)} righe nel database {db_path}.")


Nota: un altro modo per assicurarsi di non duplicare ad ogni run è semplicemente svuotare la tabella prima che questa venga popolata. In SQL si farebbe un con classico TRUNCATE TABLE; SQLite non dispone del comando perciò si fa una DELETE senza specificare i filtri.

Per i più coraggiosi, se volete, come bonus, si può provare a riscrivere il load con un DELETE come primo step e un INSERT INTO come secondo (non serve INSERT OR REPLACE dato che abbiamo troncato i dati dalla tabella).

Bonus: la domanda *"che differenza c'è tra truncate table e delete?"* è anche una di quelle domande da colloquo tecnico per figura junior


## Mettiamo tutto insieme

Finora abbiamo eseguito `extract`, `transform` e `load` in celle separate.
Scriviamo ora un'unica funzione che le richiama in sequenza: partendo solo
dalla configurazione (`elenco`, `url`, `db_path`), costruisce da sola l'intero
database. Comoda per rilanciare tutta la pipeline in un colpo solo — è anche
esattamente il codice che mettereste in un file `.py` a sé stante se voleste
eseguire l'ETL da riga di comando invece che da notebook.


**SCRIVI UNA FUNZIONE CHE UNISCE LE TRE FUNZIONI PRECEDENTEMENTE CREATE**

In [ ]:
def run_etl(elenco, url, db_path):
    """Esegue l'intera pipeline ETL: extract -> transform -> load."""


# la funzione non ha bisogno di un return, dato che il suo "return" sarebbe caricare il df nel database path

In [ ]:
run_etl(elenco, url, db_path)

---
## Interroghiamo il database con SQL

Ora che i dati sono in un database, possiamo interrogarlo con query SQL invece
di ricaricare i CSV. Ci sono due modi comuni per farlo da Python:

1. **`cursor.execute(...).fetchall()`**: esegui la query, ottieni i risultati
   come lista di tuple.
2. **`pd.read_sql(...)`**: esegui la query e ottieni direttamente un DataFrame
   pandas — più comodo quando i risultati sono tabellari.

Proviamo entrambi.


In [ ]:
conn = sqlite3.connect(db_path)

# Modo 1: cursor.execute + fetchall
cur = conn.cursor()
cur.execute("SELECT MIN(Date), MAX(Date) FROM prices")
print("Intervallo di date nel database:", cur.fetchone())

cur.execute("SELECT * FROM prices ORDER BY Date DESC LIMIT 5")
print("\nUltime 5 righe:")
for riga in cur.fetchall():
    print(riga)

conn.close()


In [ ]:
conn = sqlite3.connect(db_path)

# Modo 2: pd.read_sql -> risultato direttamente come DataFrame
media_gold_2020 = pd.read_sql(
    "SELECT AVG(GOLD) AS media_gold FROM prices WHERE Date >= '2020-01-01'",
    conn,
)
print("Prezzo medio di GOLD dal 2020 in poi:")
display(media_gold_2020)

conn.close()


### La stessa domanda, due strade

Molte domande si possono risolvere sia con una query SQL sul database, sia
direttamente con pandas sul DataFrame che avete già in memoria (dovrebbe essere `df_clean` o comunque qualsiasi sia il nome che date al `df = transform(extract(elenco, url))`.
Sotto il cofano fanno un lavoro molto simile, ciò che cambia è solo il linguaggio in cui lo scrivete.

Un esempio:


In [ ]:
conn = sqlite3.connect(db_path)

# Domanda 1: qual è il valore massimo di GOLD in tutto il periodo?
max_gold_sql = pd.read_sql("SELECT MAX(GOLD) AS max_gold FROM prices", conn).iloc[0, 0]
max_gold_pandas = df_clean["GOLD"].max()
print("Massimo GOLD - SQL:", max_gold_sql, " | pandas:", max_gold_pandas)

#iloc[0,0] serve solo per estrarre il contenuto all'interno della cella, logicamente il codice funzionerebbe anche senza

conn.close()


DatabaseError: Execution failed on sql 'SELECT MAX(GOLD) AS max_gold FROM prices': no such table: prices

# Esercizi

Rispondi alle seguenti domande:

1.   In quanti giorni WORLD ha superato i 600 punti?
2.   Qual è il valore massimo di WORLD nel 2000?
3.   Qual è il minimo valore raggiunto da GOLD tra il 04/08/2016 e il 31/12/2017?
4.   Qual è il valore medio di ciascun indice per ciascun anno?
5.   In quale anno la volatilità (deviazione standard) di WORLD_SMALL_CAP è stata più alta?
6.   Qual è la correlazione tra WORLD e GOLD?
7.   In quanti giorni WORLD ha chiuso più in alto rispetto al giorno precedente?

Nota che ora che abbiamo un DB e un modo per interrogarlo tramite python potete rispondere sia tramite python che tramite SQL. Trovo sia interessante provare ad utilizzare entrambi i metodi. Da un lato un linguaggio più semplice e "naturale" dall'altro uno più complesso ma molto potente.

Ad alcune di queste domande sarà più facile rispondere usando SQL ma man mano che si avanza con la complessità della richiesta python vince a mani basse.